# 🃏 Poker AI v4 — Kaggle Trainer (Selfplay Edition)

| GPU | Ingyenes idő | Mentés |
|---|---|---|
| T4 × 2 vagy P100 | **30 óra / hét** | Auto-push → GitHub (`models/2max/`) |

### Elindítás előtt:
- **Accelerator**: *Session Options → Accelerator → GPU T4 × 2*
- **Internet**: *Session Options → Internet → On*
- **Secrets**: `GH_TOKEN` → Kaggle Secrets (kötelező a git push-hoz!)

### Működési logika:
1. **Lépés 1** — GPU-ellenőrzés + dependenciák
2. **Lépés 2** — Mindig friss git pull/clone → **legújabb modell automatikusan betöltve**
3. **Lépés 3** — GitHub token betöltése + git remote beállítása
4. **Lépés 4** — Selfplay konfiguráció (50M ep), folytatás az aktuális checkpointtól
5. **Lépés 5** — 🚀 Tréning + automatikus git push (2 percenként)
6. **Lépés 6** — 🔚 Végső mentés és push

> **Bármikor újraindítható** — a notebook mindig a `github.com/Mikidikilang/PokerAI` repó
> `models/2max/` mappájából veszi fel a legfrissebb checkpointot és onnan folytatja.

---
## Lépés 1 — GPU-ellenőrzés és dependenciák

In [ ]:
import torch

print('=' * 52)
print('  GPU ÁLLAPOT')
print('=' * 52)
print(f'  PyTorch:  {torch.__version__}')
print(f'  CUDA:     {torch.cuda.is_available()}')

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        vram = props.total_memory / 1e9
        print(f'  GPU {i}: {props.name}  ({vram:.1f} GB VRAM)')
    print()
    total_vram = sum(
        torch.cuda.get_device_properties(i).total_memory
        for i in range(torch.cuda.device_count())
    ) / 1e9
    suggested_envs = 512 if total_vram >= 14 else 256
    print(f'  Javasolt NUM_ENVS: {suggested_envs}  (összes VRAM: {total_vram:.1f} GB)')
else:
    print()
    print('  FIGYELEM: Nincs GPU! Menj: Session Options -> Accelerator -> GPU T4 x2')

print('=' * 52)

import subprocess
subprocess.run('pip install -q rlcard>=1.0.5 tensorboard>=2.13.0', shell=True, check=True)
print('\nAlap dependenciák telepítve.')

---
## Lépés 2 — Friss kódbázis + legújabb modell gitről

**Minden indításkor** automatikusan:
- Frissíti a kódot: `git pull` (ha már klónozva) vagy `git clone`
- A `models/2max/` `.pth`, `config.json`, `naplo.json` fájlok **mindig felülíródnak gitről** — így az előző session mentése automatikusan folytatódik
- A `tests/` tartalom **additív**: meglévő snapshotok megmaradnak, a gitről hiányzók hozzáadódnak

In [ ]:
import os, shutil, subprocess, glob

REPO_URL    = 'https://github.com/Mikidikilang/PokerAI.git'
REPO_BRANCH = 'main'
MODEL_NAME  = '2max'

CODE_DIR    = '/kaggle/working/poker_ai'
OUTPUT_BASE = '/kaggle/working/PokerAI_Models'

def _run(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True)

# ── Git pull / clone ──────────────────────────────────────────────────────
if os.path.exists(os.path.join(CODE_DIR, '.git')):
    print('Repo mar megvan -> git pull (force)...')
    r = _run(f'cd {CODE_DIR} && git fetch origin {REPO_BRANCH} && git reset --hard origin/{REPO_BRANCH}')
    if r.returncode != 0:
        print('git pull hiba, ujraklonozas...')
        print(r.stderr[:400])
        shutil.rmtree(CODE_DIR, ignore_errors=True)
    else:
        commit = _run(f'cd {CODE_DIR} && git log -1 --format="%h %s"').stdout.strip()
        print(f'git pull kesz  |  {commit}')

if not os.path.exists(os.path.join(CODE_DIR, '.git')):
    print('git clone...')
    shutil.rmtree(CODE_DIR, ignore_errors=True)
    r = _run(f'git clone --branch {REPO_BRANCH} --depth 1 {REPO_URL} {CODE_DIR}')
    if r.returncode != 0:
        print('Git clone hiba:')
        print(r.stderr)
        raise RuntimeError('Git clone sikertelen! Ellenorzd az internet-hozzaferest.')
    commit = _run(f'cd {CODE_DIR} && git log -1 --format="%h %s"').stdout.strip()
    print(f'git clone kesz  |  {commit}')

# ── requirements.txt ─────────────────────────────────────────────────────
req_path = os.path.join(CODE_DIR, 'requirements.txt')
if os.path.exists(req_path):
    subprocess.run(f'pip install -q -r {req_path}', shell=True, check=True)
    print('requirements.txt dependenciak telepitve.')

# ── Legujabb modell betoltese a gitrol -> working mappaba ─────────────────
# MINDIG lefut: a notebook barhanyszor ujraindul, mindig a git legfrissebb
# checkpointjaval folytatja.
src_model_dir = os.path.join(CODE_DIR, 'models', MODEL_NAME)
dst_model_dir = os.path.join(OUTPUT_BASE, 'models', MODEL_NAME)
os.makedirs(dst_model_dir, exist_ok=True)

print(f'\nModell szinkronizalas: git -> working  ({MODEL_NAME})')

# .pth: mindig felulirja (git = legfrissebb checkpoint)
src_pths = sorted(glob.glob(os.path.join(src_model_dir, '*.pth')))
if src_pths:
    dst_pth = os.path.join(dst_model_dir, os.path.basename(src_pths[0]))
    shutil.copy2(src_pths[0], dst_pth)
    size_mb = os.path.getsize(dst_pth) / 1e6
    print(f'  .pth betoltve gitrol: {os.path.basename(dst_pth)}  ({size_mb:.1f} MB)')
else:
    print(f'  Nincs .pth a gitben ({src_model_dir}) -> Uj trening indul.')

# config.json + naplo.json: gitrol felulirja
for fname in ['config.json', 'naplo.json']:
    src = os.path.join(src_model_dir, fname)
    dst = os.path.join(dst_model_dir, fname)
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print(f'  {fname} szinkronizalva gitrol.')

# tests/: ADDITIV masolás - csak a gitben levo, helyen hianyzó mappak kerul at
# A mar meglevo helyi snapshotok erintetlenek maradnak!
src_tests = os.path.join(src_model_dir, 'tests')
dst_tests = os.path.join(dst_model_dir, 'tests')
os.makedirs(dst_tests, exist_ok=True)
if os.path.exists(src_tests):
    added = 0
    for item in os.listdir(src_tests):
        src_item = os.path.join(src_tests, item)
        dst_item = os.path.join(dst_tests, item)
        if not os.path.exists(dst_item):   # csak ha meg nincs helyben
            if os.path.isdir(src_item):
                shutil.copytree(src_item, dst_item)
            else:
                shutil.copy2(src_item, dst_item)
            added += 1
    existing = len(os.listdir(dst_tests))
    print(f'  tests/ szinkronizalva: {added} uj, osszesen {existing} snapshot.')
else:
    print('  Nincs tests/ a gitben - ures konyvtar letrehozva.')

print(f'\nFobb fajlok: {[f for f in os.listdir(CODE_DIR) if not f.startswith(".")][:10]}')

---
## Lépés 3 — GitHub Token + Git beállítás

**`GH_TOKEN` kötelező** a checkpoint auto-pushhoz.

Kaggle Secrets beállítás:
1. Kaggle → Account → **Secrets** → `+ Add New Secret`
2. Kulcs neve: `GH_TOKEN` → Értéke: GitHub Personal Access Token (`repo` scope)
3. Notebook Options → Secrets → `GH_TOKEN` ✓ engedélyezd

WandB opcionális — ugyanígy `WANDB_API_KEY` secrettel.

In [ ]:
import os, subprocess

# ── GitHub Token ──────────────────────────────────────────────────────────
GH_TOKEN = ''
try:
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()
    GH_TOKEN = _secrets.get_secret('GH_TOKEN')
    print('GH_TOKEN betoltve Kaggle Secrets-bol.')
except Exception as e:
    print(f'GH_TOKEN nem elerheto: {e}')
    print('   -> Auto git push NEM fog mukodni!')
    print('   -> Kaggle Secrets -> GH_TOKEN beallitasa szukseges.')

# ── Git remote URL beállítása tokennel ───────────────────────────────────
REPO_OWNER = 'Mikidikilang'
REPO_NAME  = 'PokerAI'

if GH_TOKEN:
    remote_url = f'https://{GH_TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git'
    for cmd in [
        f'cd {CODE_DIR} && git remote set-url origin "{remote_url}"',
        f'cd {CODE_DIR} && git config user.email "kaggle-bot@pokerai.local"',
        f'cd {CODE_DIR} && git config user.name "Kaggle Trainer"',
    ]:
        subprocess.run(cmd, shell=True, capture_output=True)
    print(f'Git remote beallitva (tokennel): {REPO_OWNER}/{REPO_NAME}')
else:
    print('GH_TOKEN hianyzik -> git remote nem allithato be tokennel.')

# ── WandB (opcionális) ────────────────────────────────────────────────────
USE_WANDB = False
if USE_WANDB:
    try:
        from kaggle_secrets import UserSecretsClient
        secrets = UserSecretsClient()
        wandb_key = secrets.get_secret('WANDB_API_KEY')
        os.environ['WANDB_API_KEY'] = wandb_key
        subprocess.run('pip install -q wandb', shell=True, check=True)
        import wandb
        wandb.login(key=wandb_key)
        print('WandB bejelentkezve.')
    except Exception as e:
        print(f'WandB bejelentkezes sikertelen: {e}')
        USE_WANDB = False
else:
    print('WandB kikapcsolva.')

---
## Lépés 4 — Selfplay konfiguráció

A modell már be van töltve (Lépés 2 szinkronizálta a gitről).
A `train.py` automatikusan folytatja onnan, ahol a checkpoint abbahagyta.

**Cél: 50 millió epizód selfplay**, 500k-ként mérföldkő + auto git push.

| Paraméter | Érték | Leírás |
|---|---|---|
| `TRAINING_MODE` | `'self_play'` | Selfplay (bot pool kikapcsolva) |
| `EPISODES` | `50_000_000` | 50M epizód (összesen, checkpointtól folytatva) |
| `SAVE_FREQ` | `500_000` | Mérföldkő 500k-ként |
| `NUM_ENVS` | `256/512` | T4: 256, P100: 512 |

In [ ]:
import os, sys, glob

# ════════════════════════════════════════════════════════════
MODEL_NAME    = '2max'        # Modell neve
NUM_PLAYERS   = 2             # Jatekosok szama
TRAINING_MODE = 'self_play'   # Selfplay mod!
EPISODES      = 50_000_000    # 50 millio epizod (teljes cel)
SAVE_FREQ     = 500_000       # Merfolddko 500k-kent
TEST_HANDS    = 500           # Teszt kezszam merfolddkonel
NUM_ENVS      = 256           # T4: 256 | P100: 512
LEARNING_RATE = 3e-4          # Adam LR
HIDDEN_SIZE   = 512           # Modell szelessege
# ════════════════════════════════════════════════════════════

MODEL_DIR = os.path.join(OUTPUT_BASE, 'models', MODEL_NAME)

if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

# ── ModelManager: config.json megírása ───────────────────────────────────
from training.model_manager import ModelManager

mgr = ModelManager(OUTPUT_BASE)
mgr.ensure_model_dir(MODEL_NAME, NUM_PLAYERS)

cfg_data = mgr.load_config(MODEL_NAME)
cfg_data['num_players'] = NUM_PLAYERS
cfg_data['config'].update({
    'num_envs':           NUM_ENVS,
    'learning_rate':      LEARNING_RATE,
    'hidden_size':        HIDDEN_SIZE,
    'milestone_interval': SAVE_FREQ,
    'milestone_hands':    TEST_HANDS,
    'training_style':     TRAINING_MODE,
})
mgr.save_config(MODEL_NAME, cfg_data)
print(f'config.json megirva  (mod: {TRAINING_MODE})')

PTH_PATH = mgr.pth_path(MODEL_NAME)

# ── Összefoglaló ──────────────────────────────────────────────────────────
print('=' * 60)
print('  KONFIGURÁCIÓ ÖSSZEFOGLALÓ')
print('=' * 60)
print(f'  Modell:         {MODEL_NAME}  ({NUM_PLAYERS} jatekos)')
print(f'  Trening mod:    {TRAINING_MODE}  (selfplay)')
print(f'  Cel epizodok:   {EPISODES:,}  (50M)')
print(f'  Merfolddko:     minden {SAVE_FREQ:,} epizodkent')
print(f'  GPU envs:       {NUM_ENVS}')
print(f'  Tanulasi rata:  {LEARNING_RATE}')
print('-' * 60)
print(f'  Output:   {OUTPUT_BASE}')
print(f'  PTH:      {PTH_PATH}')

dst_tests = os.path.join(MODEL_DIR, 'tests')
n_snaps = len([d for d in glob.glob(os.path.join(dst_tests, '*')) if os.path.isdir(d)])
print(f'  Tests:    {n_snaps} snapshot a working mappaban')
print('=' * 60)

if os.path.exists(PTH_PATH):
    from utils.checkpoint_utils import safe_load_checkpoint
    ck = safe_load_checkpoint(PTH_PATH, map_location='cpu')
    if isinstance(ck, dict) and 'episodes_trained' in ck:
        ep_so_far = ck['episodes_trained']
        remaining = max(0, EPISODES - ep_so_far)
        print(f'  Folytatas: {ep_so_far:,} epizodtol')
        print(f'  Hatramaro: {remaining:,} epizod')
    else:
        print('  Meglevo checkpoint betoltve.')
else:
    print('  Nincs checkpoint - uj trening indul.')

---
## Lépés 5 — 🚀 Tréning + automatikus git push

A tréning `train.py`-on keresztül fut, **selfplay módban, 50M epizód céllal**.
A `train.py` automatikusan folytatja a checkpoint alapján.

**Auto git push logika (háttérszál):**
- **2 percenként** ellenőrzi a `.pth` módosítási idejét
- Ha változott (= új mérföldkő mentve) → push
- **`tests/` additív**: meglévő git snapshotok megmaradnak, újak hozzáadódnak
- `.pth`, `config.json`, `naplo.json` mindig felülírja a gitbeli verziót

> **Megszakítás esetén:** legutóbbi mentés megmarad. Futtasd a 6. lépést!

In [ ]:
import subprocess, sys, os, shutil, glob, threading, time

PUSH_INTERVAL_SEC  = 120
_last_pushed_mtime = [0.0]
_stop_pusher       = [False]


def git_push_checkpoint(label='auto'):
    """Szinkronizalja a working modell mappat a git repoban, majd push."""
    repo_model_dir = os.path.join(CODE_DIR, 'models', MODEL_NAME)
    os.makedirs(repo_model_dir, exist_ok=True)

    # .pth felulirja (legfrissebb)
    if os.path.exists(PTH_PATH):
        shutil.copy2(PTH_PATH, os.path.join(repo_model_dir, os.path.basename(PTH_PATH)))

    # config.json + naplo.json felulirja
    for fname in ['config.json', 'naplo.json']:
        src = os.path.join(MODEL_DIR, fname)
        if os.path.exists(src):
            shutil.copy2(src, os.path.join(repo_model_dir, fname))

    # tests/: ADDITIV szinkronizalas - meglevo git snapshotok megmaradnak,
    # csak az ujak adodnak hozza. Semmit sem torol!
    src_tests = os.path.join(MODEL_DIR, 'tests')
    dst_tests = os.path.join(repo_model_dir, 'tests')
    os.makedirs(dst_tests, exist_ok=True)
    if os.path.exists(src_tests):
        for item in os.listdir(src_tests):
            src_item = os.path.join(src_tests, item)
            dst_item = os.path.join(dst_tests, item)
            if not os.path.exists(dst_item):    # csak az ujakat adjuk hozza
                if os.path.isdir(src_item):
                    shutil.copytree(src_item, dst_item)
                else:
                    shutil.copy2(src_item, dst_item)

    # Epizodszam a commit uzenethez
    ep_label = label
    try:
        from utils.checkpoint_utils import safe_load_checkpoint
        ck = safe_load_checkpoint(PTH_PATH, map_location='cpu')
        if isinstance(ck, dict):
            ep_label = f'ep={ck.get("episodes_trained", "?"):,}'
    except Exception:
        pass

    for cmd in [
        f'cd {CODE_DIR} && git add models/{MODEL_NAME}/',
        f'cd {CODE_DIR} && git diff --cached --quiet || git commit -m "auto: [{ep_label}] selfplay [Kaggle]"',
        f'cd {CODE_DIR} && git pull --rebase origin main -q',
        f'cd {CODE_DIR} && git push origin main',
    ]:
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        if r.returncode != 0:
            combined = (r.stdout + r.stderr).lower()
            if 'nothing to commit' in combined:
                return True
            print(f'\nGit hiba ({cmd[:45]}...): {(r.stderr or r.stdout)[:200]}')
            return False
    return True


def _pusher_thread():
    while not _stop_pusher[0]:
        time.sleep(PUSH_INTERVAL_SEC)
        if _stop_pusher[0]:
            break
        if not GH_TOKEN:
            continue
        try:
            mtime = os.path.getmtime(PTH_PATH) if os.path.exists(PTH_PATH) else 0.0
            if mtime > _last_pushed_mtime[0]:
                print(f'\n[{time.strftime("%H:%M:%S")}] Auto git push...', flush=True)
                ok = git_push_checkpoint()
                if ok:
                    _last_pushed_mtime[0] = mtime
                    print('Git push kesz!', flush=True)
                else:
                    print('Git push sikertelen. Kovetkezo kiserlet 2 perc mulva.', flush=True)
        except Exception as ex:
            print(f'\nPusher szal hiba: {ex}', flush=True)


if GH_TOKEN:
    _stop_pusher[0] = False
    _pusher = threading.Thread(target=_pusher_thread, daemon=True, name='GitPusher')
    _pusher.start()
    print(f'Git pusher szal elindult  (intervallum: {PUSH_INTERVAL_SEC}s)')
else:
    print('Git pusher kikapcsolva - GH_TOKEN hianyzik.')

cmd = [
    sys.executable,
    os.path.join(CODE_DIR, 'train.py'),
    '--model_name',  MODEL_NAME,
    '--players',     str(NUM_PLAYERS),
    '--mode',        TRAINING_MODE,
    '--iterations',  str(EPISODES),
    '--save_freq',   str(SAVE_FREQ),
    '--test_hands',  str(TEST_HANDS),
    '--output_dir',  OUTPUT_BASE,
]

print('\n' + '=' * 60)
print('  SELFPLAY TRENING INDUL - 50M epizod')
print('=' * 60)
print('  Parancs: ' + ' '.join(str(a) for a in cmd))
print('=' * 60)

env = {
    **os.environ,
    'PYTHONPATH':       CODE_DIR,
    'PYTHONUNBUFFERED': '1',
}

process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
)

try:
    for line in process.stdout:
        print(line, end='', flush=True)
    process.wait()
except KeyboardInterrupt:
    print('\nMegszakitva (Ctrl+C) - checkpoint megmarad.')
    print('   -> Futtasd a 6. lepest a vegso menteshez!')
    process.terminate()
    process.wait()
finally:
    _stop_pusher[0] = True

rc = process.returncode
if rc == 0:
    print('\nTrening sikeresen befejezve!')
elif rc is None:
    print('\nFolyamat megszakitva - checkpoint megmaradt.')
else:
    print(f'\nHiba! Visszateresi kod: {rc}')

print('-> Futtasd a 6. lepest a vegso git push-hoz!')

---
## Lépés 6 — 🔚 Végső mentés és git push

**Minden session végén futtasd** — tréning vége vagy Ctrl+C után.

- Végső push git-re: `.pth` + `config.json` + `naplo.json` + **összes** teszt snapshot
- ZIP backup a `/kaggle/working/`-ba (letölthető az Output szekcióból)

In [ ]:
import os, shutil, json, glob, sys, subprocess

if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

print('=' * 60)
print('  VEGSO CHECKPOINT ALLAPOT')
print('=' * 60)

if os.path.exists(PTH_PATH):
    from utils.checkpoint_utils import safe_load_checkpoint
    ck   = safe_load_checkpoint(PTH_PATH, map_location='cpu')
    ep   = ck.get('episodes_trained', 0) if isinstance(ck, dict) else 0
    t    = ck.get('time_spent', 0)       if isinstance(ck, dict) else 0
    size = os.path.getsize(PTH_PATH) / 1e6
    print(f'  {os.path.basename(PTH_PATH)}')
    print(f'     Epizodok:    {ep:,}  /  {EPISODES:,}')
    print(f'     Trening ido: {t/3600:.1f} ora')
    print(f'     Fajlmeret:   {size:.1f} MB')
else:
    print(f'  Nincs checkpoint: {PTH_PATH}')

# Teszt snapshots
tests_dir = os.path.join(MODEL_DIR, 'tests')
snap_dirs = sorted(
    [d for d in glob.glob(os.path.join(tests_dir, '*')) if os.path.isdir(d)]
)
print(f'\n  Teszt snapshots (osszesen {len(snap_dirs)} db - mind gitbe kerul):')
for md in snap_dirs[-5:]:
    name  = os.path.basename(md)
    grade = '?'
    for jf in glob.glob(os.path.join(md, '*.json')):
        try:
            with open(jf) as f:
                d = json.load(f)
            grade = d.get('summary', {}).get('grade', '?')
        except Exception:
            pass
    marker = '  <- legujabb' if md == snap_dirs[-1] else ''
    print(f'    {name:<40}  grade: {grade}{marker}')

# Vegso git push
print()
if GH_TOKEN and os.path.exists(PTH_PATH):
    print('Vegso git push inditas...')
    ok = git_push_checkpoint(label='final')
    if ok:
        print('Vegso git push sikeres!')
        log = subprocess.run(
            f'cd {CODE_DIR} && git log -1 --format="%h %ai %s"',
            shell=True, capture_output=True, text=True
        ).stdout.strip()
        print(f'   Commit: {log}')
    else:
        print('Vegso git push sikertelen!')
elif not GH_TOKEN:
    print('Git push kihagyva - GH_TOKEN hianyzik.')
else:
    print('Git push kihagyva - nincs checkpoint.')

# ZIP backup
zip_base = f'/kaggle/working/pokerai_{MODEL_NAME}_selfplay_checkpoint'
shutil.make_archive(zip_base, 'zip', OUTPUT_BASE)
zip_size = os.path.getsize(zip_base + '.zip') / 1e6
print(f'\n  ZIP backup: {zip_base}.zip  ({zip_size:.1f} MB)')

print()
print('=' * 60)
print('  KESZ!')
print('=' * 60)
print('  Checkpoint git-en: github.com/Mikidikilang/PokerAI')
print(f'     -> models/{MODEL_NAME}/{MODEL_NAME}_ppo_v4.pth')
print(f'     -> models/{MODEL_NAME}/tests/  ({len(snap_dirs)} snapshot)')
print()
print('  Kovetkezo session: csak inditsd ujra a notebookot.')
print('  A legfrissebb modell automatikusan betoltodik gitrol.')